# Posterior reconstruction on Wilkes3

This is a thin HPC frontend to sample-posterior. The implementation lives in src/sample.py; this notebook only configures a run and inspects its saved artifact.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
REPO = next(path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file())
print(f"Repository: {REPO}")

In [ ]:
import shlex
import subprocess
from pathlib import Path

DATA_DIR = REPO / "data" / "gals_gband_norm"
OUTPUT_DIR = REPO / "outputs" / "probes_final" / "sample_pick70"
CKPT = REPO / "outputs" / "probes_final" / "latest.pt"
RUN = False

command = [
    "sample-posterior",
    "--data_dir", str(DATA_DIR),
    "--output_dir", str(OUTPUT_DIR),
    "--ckpt", str(CKPT),
    "--steps", "8000",
    "--n_post", "160",
    "--chunk", "32",
    "--pick", "70",
    "--noise_sigma", "0.02",
    "--seed", "21",
]
print(shlex.join(command))
if RUN:
    subprocess.run(command, check=True)

Set RUN to True only inside an allocated GPU job. The equivalent batch entry point is scripts/run_sample.sh and is preferred for the final production run.

In [ ]:
import torch

artifact = OUTPUT_DIR / "samples" / "posterior_draws.pt"
if artifact.exists():
    saved = torch.load(artifact, map_location="cpu", weights_only=False)
    posterior = saved["post"].float()
    mean = posterior.mean(dim=0)
    std = posterior.std(dim=0)
    print(f"draws={posterior.shape[0]}, mean range={mean.min():.3f}..{mean.max():.3f}")
    print(f"mean posterior std={std.mean():.4f}")
else:
    print(f"Run not found yet: {artifact}")